In [1]:
!pip install --upgrade matplotlib
!pip install --upgrade seaborn
!pip install geopandas
!pip install contextily
!pip install osmnx
!pip install alphashape

In [2]:
# import libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import LineString, Point
import osmnx as ox
import networkx as nx
import numpy as np
from pyproj import Transformer
# geeksforgeeks.org/machine-learning/ball-tree-and-kd-tree-algorithms/
from sklearn.neighbors import BallTree
import requests
from shapely.geometry import MultiPoint, Point
import osmnx as ox
import alphashape
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from functools import reduce

In [5]:
# LLM helped debug
import pyproj
pyproj.datadir.set_data_dir("/opt/anaconda3/envs/envGEOG0115/lib/python3.8/site-packages/pyproj/proj_dir/share/proj")

In [6]:
CRS_BNG = "EPSG:27700"
NETWORK_TYPE = "walk"
NETWORK_BUFFER_M = 1800  # 2 miles in m
WALK_DISTANCE_M = 1609   # 1 mile in metres

MIN_REACHABLE_NODES = 4  # builds an isochrome polygon of at least 4 sides
ALPHA = 0.002            # how snuggly the polygon hugs the reachable network nodes
CACHE_DIR = Path("network_cache")
CACHE_DIR.mkdir(exist_ok=True)

In [7]:
# finalize list of schools: 
# read in schools from 2024-2025 academic year (last year in study)
schoolsForAnalysis = pd.read_csv('Data/SecondarySchools/finalSecondarySchools.csv', encoding="latin-1", low_memory=False)

# read in schools from 2016-2017 academic year (first year in study)
schoolData20162017 = pd.read_csv('Data/SecondarySchools/england_ks4final.csv', encoding='utf-8-sig', low_memory=False)

# filter to the same list so schools stay the same for the whole temporal study
schoolData20162017 = schoolData20162017[schoolData20162017['URN'].isin(schoolsForAnalysis['URN'])].copy()

secondarySchools = schoolData20162017.copy()

In [8]:
years = ["20162017", "20182019", "20212022", "20222023", "20242025"]

# Charity Commission Strict
chcStrict = {}
for year in years:
    chcStrict[year] = pd.read_csv("Data/CHCStrict/df_final_chc_strict" + year + ".csv")

# Charity Commission Broad
chcBroad = {}
for year in years:
    chcBroad[year] = pd.read_csv("Data/CHCBroad/df_final_chc_broad" + year + ".csv")

# Company House Strict
cohStrict = {}
for year in years:
    cohStrict[year] = pd.read_csv("Data/COHStrict/df_coh_final_strict" + year + ".csv")

# Company House Broad
cohBroad = {}
for year in years:
    cohBroad[year] = pd.read_csv("Data/COHBroad/df_coh_final_broad" + year + ".csv")

# Postcodes/ONSPD
onspd = pd.read_csv("Data/ONSPD/ONSPDAugust2025.csv")

/var/folders/_2/k3vltl893kjg0r81wmcvq9900000gn/T/ipykernel_27689/3055513912.py:11: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  chcBroad[year] = pd.read_csv("Data/CHCBroad/df_final_chc_broad" + year + ".csv")
/var/folders/_2/k3vltl893kjg0r81wmcvq9900000gn/T/ipykernel_27689/3055513912.py:11: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  chcBroad[year] = pd.read_csv("Data/CHCBroad/df_final_chc_broad" + year + ".csv")
/var/folders/_2/k3vltl893kjg0r81wmcvq9900000gn/T/ipykernel_27689/3055513912.py:11: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  chcBroad[year] = pd.read_csv("Data/CHCBroad/df_final_chc_broad" + year + ".csv")
/var/folders/_2/k3vltl893kjg0r81wmcvq9900000gn/T/ipykernel_27689/3055513912.py:11: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  chcBroad[year

In [1]:
# to help reduce compuational power an LLM was consulted for these steps, any functions I designed were prompted
# to speed along and certain functions I consulted to helped write in areas outside my knowledge/what was taught in our
# courses. 
class DataPrep():
    @staticmethod
    # adds lat/long coordinates to charity df using ONSPD lookup
    def addCharityCoordinates(charitydf, onspd, postcode_col='pcd2'):
        transformer = Transformer.from_crs("EPSG:4326", "EPSG:27700", always_xy=True)
        onspdFiltered = onspd[['pcd2', 'lat', 'long']]
        charitydf = charitydf.drop(columns=['lat', 'long', 'x_bng', 'y_bng'], errors='ignore')
        if postcode_col != 'pcd2':
            charitydf = charitydf.drop(columns=['pcd2'], errors='ignore')
        charitydf = charitydf.merge(onspdFiltered, left_on=postcode_col, right_on='pcd2', how='left')
        charitydf = charitydf.dropna(subset=['lat', 'long'])
        charitydf['x_bng'], charitydf['y_bng'] = transformer.transform(
            charitydf['long'].values, charitydf['lat'].values
        )
        return charitydf
    @staticmethod
    # adds lat/long coordinates to schools dataframe using Easting/Northing
    def addSchoolCoordinates(schoolsdf):
        transformer = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)
        schoolsdf = schoolsdf.dropna(subset=['Easting', 'Northing']).copy()
        schoolsdf['longitude'], schoolsdf['latitude'] = transformer.transform(
            schoolsdf['Easting'].values, schoolsdf['Northing'].values
        )
        schoolsdf['x_bng'] = schoolsdf['Easting']
        schoolsdf['y_bng'] = schoolsdf['Northing']
        return schoolsdf.dropna(subset=['latitude', 'longitude'])

class DataWrangling():
    # counts charities within a given radius of each school using BallTree
    @staticmethod
    # finds charities within a given radius of each school using BallTree,
    # and adds a column to schoolsdf listing the unique IDs of those charities
    # company_number for COH
    # registered_charity_number for CHC
    # finds charities within a given radius of each school using BallTree,
    # returns a list of unique charity IDs per school (one list per row)
    # LLM helped to debug after pyproj stopped working 
    def charitiesNearSchools(schoolsdf, charitydf, id_col, radius_miles=1):
        if 'lat' not in charitydf.columns or 'long' not in charitydf.columns:
            charitydf = DataPrep.addCharityCoordinates(charitydf)
        charitydf = charitydf.dropna(subset=['lat', 'long', 'x_bng', 'y_bng'])
        charitydf = charitydf[np.isfinite(charitydf['lat']) & np.isfinite(charitydf['long'])]
    
        school_coords = np.radians(schoolsdf[['latitude', 'longitude']].values)
        charity_coords = np.radians(charitydf[['lat', 'long']].values)
    
        tree = BallTree(charity_coords, leaf_size=40, metric='haversine')
        radius = (radius_miles * 1609) / 6371000
    
        indices = tree.query_radius(school_coords, r=radius)
    
        nearby_id_lists = []
        for pos in range(len(schoolsdf)):
            matches = charitydf.iloc[indices[pos]]
            nearby_id_lists.append(matches[id_col].dropna().unique().tolist())
        return nearby_id_lists

    # downloads the OSM network 
    @staticmethod
    def downloadSchoolNetwork(lat: float, lon: float, est_num):
        cache_path = CACHE_DIR / f"network_{est_num}.graphml"

        if cache_path.exists():
            return ox.load_graphml(cache_path)

        G = ox.graph_from_point(
            (lat, lon),
            dist=NETWORK_BUFFER_M,   # 3218 m = 2 miles
            network_type=NETWORK_TYPE,
            simplify=True
        )
        G = ox.project_graph(G, to_crs=CRS_BNG)
        ox.save_graphml(G, cache_path)
        return G

    @staticmethod
    # LLM helped write/debug
    # snaps any point (school or charity) to its nearest network node
    def snapToNetwork(G, x_bng, y_bng):
        return ox.distance.nearest_nodes(G, X=x_bng, Y=y_bng)

    @staticmethod
    # finds every node reachable within the walk distance, as a set for fast lookup
    def getReachableNodes(G, school_node, walk_distance_m=WALK_DISTANCE_M):
        lengths = nx.single_source_dijkstra_path_length(
            G, school_node, cutoff=walk_distance_m, weight="length"
        )
        return set(lengths.keys())

    @staticmethod
    # checks whether ONE charity is within walking distance of the school
    def checkDistance(G, charity, reachable_nodes):
        charity_node = DataWrangling.snapToNetwork(G, charity['x_bng'], charity['y_bng'])
        return charity_node in reachable_nodes

    @staticmethod
    # counts how many of a school's nearby charities are walkable
    def countWalkableCharities(G, school, charitydf, nearby_ids, id_col,walk_distance_m=WALK_DISTANCE_M):
        if len(nearby_ids) == 0:
            return 0
        school_node = DataWrangling.snapToNetwork(G, school['x_bng'], school['y_bng'])
        reachable = DataWrangling.getReachableNodes(G, school_node, walk_distance_m)
        nearby = charitydf[charitydf[id_col].isin(nearby_ids)]
        count = 0
        for _, charity in nearby.iterrows():
            if DataWrangling.checkDistance(G, charity, reachable):
                count += 1
        return count

    @staticmethod
    # loops through all schools and fills the numWalkableCharities column
    def addWalkableCharityCounts(schoolsdf, charitydf, id_col, walk_distance_m=WALK_DISTANCE_M):
        schoolsdf = schoolsdf.copy()
        schoolsdf['numWalkableCharities'] = pd.NA
        for index, school in tqdm(schoolsdf.iterrows(), total=len(schoolsdf)):
            nearby = school['nearbyCharities']
            # skip download entirely if nothing is nearby
            if nearby is None or len(nearby) == 0:
                schoolsdf.at[index, 'numWalkableCharities'] = 0
                continue
            try:
                G = DataWrangling.downloadSchoolNetwork(
                    school['latitude'], school['longitude'], school['EstablishmentNumber']
                )
                schoolsdf.at[index, 'numWalkableCharities'] = \
                    DataWrangling.countWalkableCharities(
                        G, school, charitydf, nearby, id_col, walk_distance_m
                    )
            except Exception as e:
                print(f"Failed: {school['EstablishmentName']} — {e}")
        return schoolsdf

NameError: name 'WALK_DISTANCE_M' is not defined

In [29]:
secondarySchools = DataPrep.addSchoolCoordinates(secondarySchools)

In [30]:
years = ["20162017", "20182019", "20212022", "20222023", "20242025"]

for year in years:
    chcStrict[year] = DataPrep.addCharityCoordinates(chcStrict[year], onspd, postcode_col='pcd2')
    chcBroad[year]  = DataPrep.addCharityCoordinates(chcBroad[year], onspd, postcode_col='pcd2')
    cohStrict[year] = DataPrep.addCharityCoordinates(cohStrict[year], onspd, postcode_col='pcd2')
    cohBroad[year]  = DataPrep.addCharityCoordinates(cohBroad[year], onspd, postcode_col='pcd2')

In [31]:
print(secondarySchools['GOR'].unique())

['London' 'West Midlands' 'North West' 'Yorkshire and the Humber'
 'North East' 'South West' 'East of England' 'South East' 'East Midlands']


In [32]:
years = ["20162017", "20182019", "20212022", "20222023", "20242025"]
definitions = {
    'CHCStrict': (chcStrict, 'registered_charity_number'),
    'CHCBroad':  (chcBroad,  'registered_charity_number'),
    'COHStrict': (cohStrict, 'company_number'),
    'COHBroad':  (cohBroad,  'company_number'),
}

## LONDON

In [ ]:
# LLM helped breakdown the following steps to stop work from being lost after running for hours and then failing 
import osmnx as ox
try:
    G = ox.graph_from_point((52.48, -1.9), dist=500, network_type='walk')
    print("Overpass OK:", len(G.nodes), "nodes")
except Exception as e:
    print("Overpass blocked:", e)

In [ ]:
import time

# STEP 1: one network + reachable set per school (with retry for transient Overpass failures)
londonSchools = secondarySchools[
    (secondarySchools['GOR'] == 'London'))
].copy()

school_cache = {}
for _, school in tqdm(westMidlandsNorthWestSchools.iterrows(), total=len(westMidlandsNorthWestSchools)):
    success = False
    for attempt in range(3):
        try:
            G = DataWrangling.downloadSchoolNetwork(
                school['latitude'], school['longitude'], school['EstablishmentNumber']
            )
            node = DataWrangling.snapToNetwork(G, school['x_bng'], school['y_bng'])
            reachable = DataWrangling.getReachableNodes(G, node)
            school_cache[school['URN']] = (G, reachable)
            success = True
            break
        except Exception as e:
            if attempt < 2:
                time.sleep(10)   # pause, then retry
            else:
                print(f"Failed {school['EstablishmentName']}: {e}")
    if not success:
        school_cache[school['URN']] = None

In [48]:
# STEP 2: reuse cached networks for every definition AND year (no re-download)
for name, (charity_dict, id_col) in definitions.items():
    for year in years:
        charity_df = charity_dict[year]
        df = londonSchools.copy()
        df['nearbyCharities'] = DataWrangling.charitiesNearSchools(df, charity_df, id_col, radius_miles=1)
        counts = []
        for _, school in df.iterrows():
            entry = school_cache.get(school['URN'])
            if entry is None:
                counts.append(pd.NA); continue
            G, reachable = entry
            nearby = charity_df[charity_df[id_col].isin(school['nearbyCharities'])]
            if len(nearby) == 0:
                counts.append(0); continue
            ch_nodes = ox.distance.nearest_nodes(G, X=nearby['x_bng'].values, Y=nearby['y_bng'].values)
            counts.append(sum(n in reachable for n in np.atleast_1d(ch_nodes)))
        df['numWalkableCharities'] = counts
        df.to_csv(f"Data/London/londonSchools_{name}_{year}.csv", index=False)
        print(f"Done: {name} {year}")

Done: CHCStrict 20162017
Done: CHCStrict 20182019
Done: CHCStrict 20212022
Done: CHCStrict 20222023
Done: CHCStrict 20242025
Done: CHCBroad 20162017
Done: CHCBroad 20182019
Done: CHCBroad 20212022
Done: CHCBroad 20222023
Done: CHCBroad 20242025
Done: COHStrict 20162017
Done: COHStrict 20182019
Done: COHStrict 20212022
Done: COHStrict 20222023
Done: COHStrict 20242025
Done: COHBroad 20162017
Done: COHBroad 20182019
Done: COHBroad 20212022
Done: COHBroad 20222023
Done: COHBroad 20242025


## West Midlands AND North West

In [11]:
import osmnx as ox
try:
    G = ox.graph_from_point((52.48, -1.9), dist=500, network_type='walk')
    print("Overpass OK:", len(G.nodes), "nodes")
except Exception as e:
    print("Overpass blocked:", e)

Overpass OK: 949 nodes


In [12]:
import time

# STEP 1: one network + reachable set per school (with retry for transient Overpass failures)
westMidlandsNorthWestSchools = secondarySchools[
    (secondarySchools['GOR'] == 'West Midlands') | (secondarySchools['GOR'] == 'North West')
].copy()

school_cache = {}
for _, school in tqdm(westMidlandsNorthWestSchools.iterrows(), total=len(westMidlandsNorthWestSchools)):
    success = False
    for attempt in range(3):
        try:
            G = DataWrangling.downloadSchoolNetwork(
                school['latitude'], school['longitude'], school['EstablishmentNumber']
            )
            node = DataWrangling.snapToNetwork(G, school['x_bng'], school['y_bng'])
            reachable = DataWrangling.getReachableNodes(G, node)
            school_cache[school['URN']] = (G, reachable)
            success = True
            break
        except Exception as e:
            if attempt < 2:
                time.sleep(10)   # pause, then retry
            else:
                print(f"Failed {school['EstablishmentName']}: {e}")
    if not success:
        school_cache[school['URN']] = None

  3%|█                                         | 15/589 [00:36<46:18,  4.84s/it]

Failed Yardleys School: local variable 'response' referenced before assignment


  5%|█▉                                      | 28/589 [05:00<4:26:54, 28.55s/it]

Failed Broadway Academy: local variable 'response' referenced before assignment


  8%|███                                     | 46/589 [05:48<1:03:03,  6.97s/it]

...

 83%|██████████████████████████████████▏      | 491/589 [42:17<20:56, 12.82s/it]

Failed The Priory School, A Business and Enterprise College: local variable 'response' referenced before assignment


 84%|██████████████████████████████████▎      | 493/589 [42:40<20:42, 12.94s/it]

Failed The Lacon Childe School: local variable 'response' referenced before assignment


100%|█████████████████████████████████████████| 589/589 [49:25<00:00,  5.04s/it]


In [17]:
# STEP 2: reuse cached networks for every definition AND year (no re-download)
for name, (charity_dict, id_col) in definitions.items():
    for year in years:
        charity_df = charity_dict[year]
        df = westMidlandsNorthWestSchools.copy()
        df['nearbyCharities'] = DataWrangling.charitiesNearSchools(df, charity_df, id_col, radius_miles=1)
        counts = []
        for _, school in df.iterrows():
            entry = school_cache.get(school['URN'])
            if entry is None:
                counts.append(pd.NA); continue
            G, reachable = entry
            nearby = charity_df[charity_df[id_col].isin(school['nearbyCharities'])]
            if len(nearby) == 0:
                counts.append(0); continue
            ch_nodes = ox.distance.nearest_nodes(G, X=nearby['x_bng'].values, Y=nearby['y_bng'].values)
            counts.append(sum(n in reachable for n in np.atleast_1d(ch_nodes)))
        df['numWalkableCharities'] = counts
        df.to_csv(f"Data/WestMidlands-NorthWest/westMidlandsNorthWestSchools_{name}_{year}.csv", index=False)
        print(f"Done: {name} {year}")

Done: CHCStrict 20162017
Done: CHCStrict 20182019
Done: CHCStrict 20212022
Done: CHCStrict 20222023
Done: CHCStrict 20242025
Done: CHCBroad 20162017
Done: CHCBroad 20182019
Done: CHCBroad 20212022
Done: CHCBroad 20222023
Done: CHCBroad 20242025
Done: COHStrict 20162017
Done: COHStrict 20182019
Done: COHStrict 20212022
Done: COHStrict 20222023
Done: COHStrict 20242025
Done: COHBroad 20162017
Done: COHBroad 20182019
Done: COHBroad 20212022
Done: COHBroad 20222023
Done: COHBroad 20242025


### Yorkshire and the Humber and North East and South West

In [19]:
import time

# STEP 1: one network + reachable set per school (with retry for transient Overpass failures)
YorkshireNorthEastSouthWestSchools = secondarySchools[
    (secondarySchools['GOR'] == 'Yorkshire and the Humber') | (secondarySchools['GOR'] == 'North East') |  (secondarySchools['GOR'] == 'South West')
].copy()

school_cache = {}
for _, school in tqdm(YorkshireNorthEastSouthWestSchools.iterrows(), total=len(YorkshireNorthEastSouthWestSchools)):
    success = False
    for attempt in range(3):
        try:
            G = DataWrangling.downloadSchoolNetwork(
                school['latitude'], school['longitude'], school['EstablishmentNumber']
            )
            node = DataWrangling.snapToNetwork(G, school['x_bng'], school['y_bng'])
            reachable = DataWrangling.getReachableNodes(G, node)
            school_cache[school['URN']] = (G, reachable)
            success = True
            break
        except Exception as e:
            if attempt < 2:
                time.sleep(10)   # pause, then retry
            else:
                print(f"Failed {school['EstablishmentName']}: {e}")
    if not success:
        school_cache[school['URN']] = None

 11%|████▋                                     | 59/535 [05:58<44:27,  5.60s/it]

Failed Bradford Girls' Grammar School: local variable 'response' referenced before assignment


 23%|█████████▌                               | 124/535 [07:41<46:46,  6.83s/it]

Failed The Co-operative Academy of Leeds: local variable 'response' referenced before assignment


 29%|███████████▎                           | 155/535 [10:02<1:20:51, 12.77s/it]

...

Failed Cramlington Learning Village: local variable 'response' referenced before assignment


 98%|████████████████████████████████████████ | 522/535 [31:27<03:46, 17.43s/it]

Failed Stanchester Academy: local variable 'response' referenced before assignment


100%|█████████████████████████████████████████| 535/535 [34:03<00:00,  3.82s/it]

Failed St Dunstan's School: local variable 'response' referenced before assignment


In [22]:
# STEP 2: reuse cached networks for every definition AND year (no re-download)
for name, (charity_dict, id_col) in definitions.items():
    for year in years:
        charity_df = charity_dict[year]
        df = YorkshireNorthEastSouthWestSchools.copy()
        df['nearbyCharities'] = DataWrangling.charitiesNearSchools(df, charity_df, id_col, radius_miles=1)
        counts = []
        for _, school in df.iterrows():
            entry = school_cache.get(school['URN'])
            if entry is None:
                counts.append(pd.NA); continue
            G, reachable = entry
            nearby = charity_df[charity_df[id_col].isin(school['nearbyCharities'])]
            if len(nearby) == 0:
                counts.append(0); continue
            ch_nodes = ox.distance.nearest_nodes(G, X=nearby['x_bng'].values, Y=nearby['y_bng'].values)
            counts.append(sum(n in reachable for n in np.atleast_1d(ch_nodes)))
        df['numWalkableCharities'] = counts
        df.to_csv(f"Data/Yorkshire-NorthEast-SouthWest/YorkshireNorthEastSouthWestSchools{name}_{year}.csv", index=False)
        print(f"Done: {name} {year}")

Done: CHCStrict 20162017
Done: CHCStrict 20182019
Done: CHCStrict 20212022
Done: CHCStrict 20222023
Done: CHCStrict 20242025
Done: CHCBroad 20162017
Done: CHCBroad 20182019
Done: CHCBroad 20212022
Done: CHCBroad 20222023
Done: CHCBroad 20242025
Done: COHStrict 20162017
Done: COHStrict 20182019
Done: COHStrict 20212022
Done: COHStrict 20222023
Done: COHStrict 20242025
Done: COHBroad 20162017
Done: COHBroad 20182019
Done: COHBroad 20212022
Done: COHBroad 20222023
Done: COHBroad 20242025


### 'East of England' , 'South East', 'East Midlands'

In [34]:
import time

# STEP 1: one network + reachable set per school (with retry for transient Overpass failures)
eastEnglandSouthEastEastMidlandsSchools = secondarySchools[
    (secondarySchools['GOR'] == 'East of England') | (secondarySchools['GOR'] == 'South East') |  (secondarySchools['GOR'] == 'East Midlands')
].copy()

school_cache = {}
for _, school in tqdm(eastEnglandSouthEastEastMidlandsSchools.iterrows(), total=len(eastEnglandSouthEastEastMidlandsSchools)):
    success = False
    for attempt in range(3):
        try:
            G = DataWrangling.downloadSchoolNetwork(
                school['latitude'], school['longitude'], school['EstablishmentNumber']
            )
            node = DataWrangling.snapToNetwork(G, school['x_bng'], school['y_bng'])
            reachable = DataWrangling.getReachableNodes(G, node)
            school_cache[school['URN']] = (G, reachable)
            success = True
            break
        except Exception as e:
            if attempt < 2:
                time.sleep(10)   # pause, then retry
            else:
                print(f"Failed {school['EstablishmentName']}: {e}")
    if not success:
        school_cache[school['URN']] = None

 46%|██████████████████▋                      | 368/806 [08:31<48:36,  6.66s/it]

Failed Burnt Mill Academy: local variable 'response' referenced before assignment


 47%|██████████████████▏                    | 377/806 [12:35<2:44:18, 22.98s/it]

Failed St Bernard's High School: local variable 'response' referenced before assignment


 47%|██████████████████▍                    | 380/806 [13:07<1:55:16, 16.24s/it]

...

 96%|███████████████████████████████████████▎ | 773/806 [40:24<03:58,  7.24s/it]

Failed Thomas Knyvett College: local variable 'response' referenced before assignment


 99%|████████████████████████████████████████▍| 794/806 [41:47<01:12,  6.08s/it]

Failed Durrington High School: local variable 'response' referenced before assignment


100%|█████████████████████████████████████████| 806/806 [42:42<00:00,  3.18s/it]


In [36]:
# STEP 2: reuse cached networks for every definition AND year (no re-download)
for name, (charity_dict, id_col) in definitions.items():
    for year in years:
        charity_df = charity_dict[year]
        df = eastEnglandSouthEastEastMidlandsSchools.copy()
        df['nearbyCharities'] = DataWrangling.charitiesNearSchools(df, charity_df, id_col, radius_miles=1)
        counts = []
        for _, school in df.iterrows():
            entry = school_cache.get(school['URN'])
            if entry is None:
                counts.append(pd.NA); continue
            G, reachable = entry
            nearby = charity_df[charity_df[id_col].isin(school['nearbyCharities'])]
            if len(nearby) == 0:
                counts.append(0); continue
            ch_nodes = ox.distance.nearest_nodes(G, X=nearby['x_bng'].values, Y=nearby['y_bng'].values)
            counts.append(sum(n in reachable for n in np.atleast_1d(ch_nodes)))
        df['numWalkableCharities'] = counts
        df.to_csv(f"Data/EastEngland-SouthEast-EastMidlands/eastEnglandSouthEastEastMidlandsSchools{name}_{year}.csv", index=False)
        print(f"Done: {name} {year}")

Done: CHCStrict 20162017
Done: CHCStrict 20182019
Done: CHCStrict 20212022
Done: CHCStrict 20222023
Done: CHCStrict 20242025
Done: CHCBroad 20162017
Done: CHCBroad 20182019
Done: CHCBroad 20212022
Done: CHCBroad 20222023
Done: CHCBroad 20242025
Done: COHStrict 20162017
Done: COHStrict 20182019
Done: COHStrict 20212022
Done: COHStrict 20222023
Done: COHStrict 20242025
Done: COHBroad 20162017
Done: COHBroad 20182019
Done: COHBroad 20212022
Done: COHBroad 20222023
Done: COHBroad 20242025


### 2016 - 2017

In [55]:
def load_region_dfs(measure, years="20162017"):
    base = "Data"
    files = [
        f"{base}/London/londonSchools_{measure}_{years}.csv",
        f"{base}/WestMidlands-NorthWest/westMidlandsNorthWestSchools_{measure}_{years}.csv",
        f"{base}/Yorkshire-NorthEast-SouthWest/YorkshireNorthEastSouthWestSchools{measure}_{years}.csv",
        f"{base}/EastEngland-SouthEast-EastMidlands/eastEnglandSouthEastEastMidlandsSchools{measure}_{years}.csv",
    ]
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

CHCStrictDfs20162017 = load_region_dfs("CHCStrict")
CHCBroadDfs20162017  = load_region_dfs("CHCBroad")
COHStrictDfs20162017 = load_region_dfs("COHStrict")
COHBroadDfs20162017  = load_region_dfs("COHBroad")

In [58]:
col = 'StudentsAchievingGrade5PlusEngMathRate'
CHCStrictDfs20162017 = CHCStrictDfs20162017.dropna(subset=[col])
CHCBroadDfs20162017  = CHCBroadDfs20162017.dropna(subset=[col])
COHStrictDfs20162017 = COHStrictDfs20162017.dropna(subset=[col])
COHBroadDfs20162017  = COHBroadDfs20162017.dropna(subset=[col])

dfs = [
    CHCStrictDfs20162017[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'CHCStrictNumWalkable'}),
    CHCBroadDfs20162017[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'CHCBroadNumWalkable'}),
    COHStrictDfs20162017[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'COHStrictNumWalkable'}),
    COHBroadDfs20162017[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'COHBroadNumWalkable'}),
]
borda = reduce(lambda a, b: a.merge(b, on='URN', how='inner'), dfs)

count_cols = ['CHCStrictNumWalkable', 'CHCBroadNumWalkable', 'COHStrictNumWalkable', 'COHBroadNumWalkable']

borda[count_cols] = borda[count_cols].fillna(0)

for c in count_cols:
    borda[f'{c}_rank'] = borda[c].rank(ascending=False, method='min')

rank_cols = [f'{c}_rank' for c in count_cols]
borda['borda_total'] = borda[rank_cols].sum(axis=1)
borda['borda_rank'] = borda['borda_total'].rank(ascending=True, method='min')

borda = borda.merge(CHCStrictDfs20162017.drop(columns=['numWalkableCharities']), on='URN', how='left')

N = len(borda)
for c in count_cols:
    borda[f'{c}_points'] = N - borda[f'{c}_rank']

points_cols = [f'{c}_points' for c in count_cols]
borda['borda_points_total'] = borda[points_cols].sum(axis=1)

print('borda_total NaN:', borda['borda_total'].isna().sum())
print('borda_rank NaN:', borda['borda_rank'].isna().sum())
print('borda_points_total NaN:', borda['borda_points_total'].isna().sum())

borda_total NaN: 0
borda_rank NaN: 0
borda_points_total NaN: 0


In [59]:
FINALDF = borda[['URN', 'EstablishmentName', 'StudentsAchievingGrade5PlusEngMathRate',
                 'borda_rank', 'borda_points_total', 'NumberOfPupils', 'PercentOfStudentReceivingFreeSchoolMeals',
                 'UrbanRuralClassification', 'GOR', 'LA (code)', 'LA (name)',
                 'LSOA (code)', 'MSOA (code)', 'Postcode',
                 'longitude', 'latitude', 'Easting', 'Northing']]
FINALDF = FINALDF.sort_values('borda_rank').reset_index(drop=True)
FINALDF.to_csv(f"Data/20162017/finalDf20162017.csv", index=False)  

In [60]:
FINALDF.head()

,URN,EstablishmentName,StudentsAchievingGrade5PlusEngMathRate,borda_rank,borda_points_total,NumberOfPupils,PercentOfStudentReceivingFreeSchoolMeals,UrbanRuralClassification,GOR,LA (code),LA (name),LSOA (code),MSOA (code),Postcode,longitude,latitude,Easting,Northing
0,100849.0,St Saviour's and St Olave's Church of England ...,57.0,1.0,9995.0,125.0,48.0,Urban: Nearer to a major town or city,London,210,Southwark,E01003942,E02000812,SE1 4AN,-0.087552,51.494665,532857.0,179036.0
1,100458.0,Central Foundation Boys' School,71.0,2.0,9953.0,135.0,59.0,Urban: Nearer to a major town or city,London,206,Islington,E01002704,E02000576,EC2A 4SH,-0.085823,51.525257,532888.0,182441.0
2,100282.0,Our Lady's Convent Roman Catholic High School,49.0,3.0,9949.0,121.0,48.0,Urban: Nearer to a major town or city,London,204,Hackney,E01001809,E02000345,N16 5AF,-0.075207,51.574273,533481.0,187911.0
3,100055.0,Maria Fidelis Roman Catholic Convent School FCJ,40.0,4.0,9948.0,78.0,62.0,Urban: Nearer to a major town or city,London,202,Camden,E01000955,E02000187,NW1 1LY,-0.132428,51.529894,529642.0,182873.0
4,133599.0,Yesodey Hatorah Senior Girls School,68.0,5.0,9947.0,56.0,7.0,Urban: Nearer to a major town or city,London,204,Hackney,E01001826,E02000348,N16 6UB,-0.068341,51.573522,533959.0,187840.0


### 2018 - 2019

In [61]:
def load_region_dfs(measure, years="20182019"):
    base = "Data"
    files = [
        f"{base}/London/londonSchools_{measure}_{years}.csv",
        f"{base}/WestMidlands-NorthWest/westMidlandsNorthWestSchools_{measure}_{years}.csv",
        f"{base}/Yorkshire-NorthEast-SouthWest/YorkshireNorthEastSouthWestSchools{measure}_{years}.csv",
        f"{base}/EastEngland-SouthEast-EastMidlands/eastEnglandSouthEastEastMidlandsSchools{measure}_{years}.csv",
    ]
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

CHCStrictDfs20182019 = load_region_dfs("CHCStrict")
CHCBroadDfs20182019  = load_region_dfs("CHCBroad")
COHStrictDfs20182019 = load_region_dfs("COHStrict")
COHBroadDfs20182019  = load_region_dfs("COHBroad")

In [62]:
col = 'StudentsAchievingGrade5PlusEngMathRate'
CHCStrictDfs20182019 = CHCStrictDfs20182019.dropna(subset=[col])
CHCBroadDfs20182019  = CHCBroadDfs20182019.dropna(subset=[col])
COHStrictDfs20182019 = COHStrictDfs20182019.dropna(subset=[col])
COHBroadDfs20182019  = COHBroadDfs20182019.dropna(subset=[col])

dfs = [
    CHCStrictDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'CHCStrictNumWalkable'}),
    CHCBroadDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'CHCBroadNumWalkable'}),
    COHStrictDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'COHStrictNumWalkable'}),
    COHBroadDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'COHBroadNumWalkable'}),
]
borda = reduce(lambda a, b: a.merge(b, on='URN', how='inner'), dfs)

count_cols = ['CHCStrictNumWalkable', 'CHCBroadNumWalkable', 'COHStrictNumWalkable', 'COHBroadNumWalkable']

borda[count_cols] = borda[count_cols].fillna(0)

for c in count_cols:
    borda[f'{c}_rank'] = borda[c].rank(ascending=False, method='min')

rank_cols = [f'{c}_rank' for c in count_cols]
borda['borda_total'] = borda[rank_cols].sum(axis=1)
borda['borda_rank'] = borda['borda_total'].rank(ascending=True, method='min')

borda = borda.merge(CHCStrictDfs20182019.drop(columns=['numWalkableCharities']), on='URN', how='left')

N = len(borda)
for c in count_cols:
    borda[f'{c}_points'] = N - borda[f'{c}_rank']

points_cols = [f'{c}_points' for c in count_cols]
borda['borda_points_total'] = borda[points_cols].sum(axis=1)

In [63]:
FINALDF = borda[['URN', 'EstablishmentName', 'StudentsAchievingGrade5PlusEngMathRate',
                 'borda_rank', 'borda_points_total', 'NumberOfPupils', 'PercentOfStudentReceivingFreeSchoolMeals',
                 'UrbanRuralClassification', 'GOR', 'LA (code)', 'LA (name)',
                 'LSOA (code)', 'MSOA (code)', 'Postcode',
                 'longitude', 'latitude', 'Easting', 'Northing']]
FINALDF = FINALDF.sort_values('borda_rank').reset_index(drop=True)
FINALDF.to_csv(f"Data/20182019/finalDf20182019.csv", index=False)  

### 2021 - 2022

In [74]:
CHCStrictDfs20212022 = load_region_dfs("CHCStrict", "20212022")
CHCBroadDfs20212022  = load_region_dfs("CHCBroad", "20212022")
COHStrictDfs20212022 = load_region_dfs("COHStrict", "20212022")
COHBroadDfs20212022  = load_region_dfs("COHBroad", "20212022")

In [75]:
col = 'StudentsAchievingGrade5PlusEngMathRate'
CHCStrictDfs20212022 = CHCStrictDfs20212022.dropna(subset=[col])
CHCBroadDfs20212022  = CHCBroadDfs20212022.dropna(subset=[col])
COHStrictDfs20212022 = COHStrictDfs20212022.dropna(subset=[col])
COHBroadDfs20212022  = COHBroadDfs20212022.dropna(subset=[col])

dfs = [
    CHCStrictDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'CHCStrictNumWalkable'}),
    CHCBroadDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'CHCBroadNumWalkable'}),
    COHStrictDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'COHStrictNumWalkable'}),
    COHBroadDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'COHBroadNumWalkable'}),
]
borda = reduce(lambda a, b: a.merge(b, on='URN', how='inner'), dfs)

count_cols = ['CHCStrictNumWalkable', 'CHCBroadNumWalkable', 'COHStrictNumWalkable', 'COHBroadNumWalkable']

borda[count_cols] = borda[count_cols].fillna(0)

for c in count_cols:
    borda[f'{c}_rank'] = borda[c].rank(ascending=False, method='min')

rank_cols = [f'{c}_rank' for c in count_cols]
borda['borda_total'] = borda[rank_cols].sum(axis=1)
borda['borda_rank'] = borda['borda_total'].rank(ascending=True, method='min')

borda = borda.merge(CHCStrictDfs20182019.drop(columns=['numWalkableCharities']), on='URN', how='left')

N = len(borda)
for c in count_cols:
    borda[f'{c}_points'] = N - borda[f'{c}_rank']

points_cols = [f'{c}_points' for c in count_cols]
borda['borda_points_total'] = borda[points_cols].sum(axis=1)

In [76]:
FINALDF = borda[['URN', 'EstablishmentName', 'StudentsAchievingGrade5PlusEngMathRate',
                 'borda_rank', 'borda_points_total', 'NumberOfPupils', 'PercentOfStudentReceivingFreeSchoolMeals',
                 'UrbanRuralClassification', 'GOR', 'LA (code)', 'LA (name)',
                 'LSOA (code)', 'MSOA (code)', 'Postcode',
                 'longitude', 'latitude', 'Easting', 'Northing']]
FINALDF = FINALDF.sort_values('borda_rank').reset_index(drop=True)
FINALDF.to_csv(f"Data/20212022/finalDf20212022.csv", index=False)

In [78]:
truly_missing = borda[count_cols].eq(0).all(axis=1)
print(truly_missing.sum(), 'of', len(borda), 'schools are 0 across all 4 lists')

# check which MSOAs are hitting the exact min value
min_val = msoa_borda['MSOA_BordaPointsMean'].min()
pileup_msoas = msoa_borda[msoa_borda['MSOA_BordaPointsMean'] == min_val]['MSOA (code)']
schools_per_pileup_msoa = schoolData20242025[schoolData20242025['MSOA (code)'].isin(pileup_msoas)].groupby('MSOA (code)').size()
print(schools_per_pileup_msoa.describe())

474 of 2509 schools are 0 across all 4 lists


NameError: name 'msoa_borda' is not defined

### 2022 - 2023

In [67]:
CHCStrictDfs20222023 = load_region_dfs("CHCStrict", "20222023")
CHCBroadDfs20222023  = load_region_dfs("CHCBroad", "20222023")
COHStrictDfs20222023 = load_region_dfs("COHStrict", "20222023")
COHBroadDfs20222023  = load_region_dfs("COHBroad", "20222023")

In [68]:
col = 'StudentsAchievingGrade5PlusEngMathRate'
CHCStrictDfs20222023 = CHCStrictDfs20222023.dropna(subset=[col])
CHCBroadDfs20222023  = CHCBroadDfs20222023.dropna(subset=[col])
COHStrictDfs20222023 = COHStrictDfs20222023.dropna(subset=[col])
COHBroadDfs20222023  = COHBroadDfs20222023.dropna(subset=[col])

dfs = [
    CHCStrictDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'CHCStrictNumWalkable'}),
    CHCBroadDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'CHCBroadNumWalkable'}),
    COHStrictDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'COHStrictNumWalkable'}),
    COHBroadDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'COHBroadNumWalkable'}),
]
borda = reduce(lambda a, b: a.merge(b, on='URN', how='inner'), dfs)

count_cols = ['CHCStrictNumWalkable', 'CHCBroadNumWalkable', 'COHStrictNumWalkable', 'COHBroadNumWalkable']

borda[count_cols] = borda[count_cols].fillna(0)

for c in count_cols:
    borda[f'{c}_rank'] = borda[c].rank(ascending=False, method='min')

rank_cols = [f'{c}_rank' for c in count_cols]
borda['borda_total'] = borda[rank_cols].sum(axis=1)
borda['borda_rank'] = borda['borda_total'].rank(ascending=True, method='min')

borda = borda.merge(CHCStrictDfs20182019.drop(columns=['numWalkableCharities']), on='URN', how='left')

N = len(borda)
for c in count_cols:
    borda[f'{c}_points'] = N - borda[f'{c}_rank']

points_cols = [f'{c}_points' for c in count_cols]
borda['borda_points_total'] = borda[points_cols].sum(axis=1)

In [69]:
FINALDF = borda[['URN', 'EstablishmentName', 'StudentsAchievingGrade5PlusEngMathRate',
                 'borda_rank', 'borda_points_total', 'NumberOfPupils', 'PercentOfStudentReceivingFreeSchoolMeals',
                 'UrbanRuralClassification', 'GOR', 'LA (code)', 'LA (name)',
                 'LSOA (code)', 'MSOA (code)', 'Postcode',
                 'longitude', 'latitude', 'Easting', 'Northing']]
FINALDF = FINALDF.sort_values('borda_rank').reset_index(drop=True)
FINALDF.to_csv(f"Data/20222023/finalDf20222023.csv", index=False)

### 2024 - 2025

In [70]:
CHCStrictDfs20242025 = load_region_dfs("CHCStrict", "20242025")
CHCBroadDfs20242025  = load_region_dfs("CHCBroad", "20242025")
COHStrictDfs20242025 = load_region_dfs("COHStrict", "20242025")
COHBroadDfs20242025  = load_region_dfs("COHBroad", "20242025")

In [71]:
col = 'StudentsAchievingGrade5PlusEngMathRate'
CHCStrictDfs20242025 = CHCStrictDfs20242025.dropna(subset=[col])
CHCBroadDfs20242025  = CHCBroadDfs20242025.dropna(subset=[col])
COHStrictDfs20242025 = COHStrictDfs20242025.dropna(subset=[col])
COHBroadDfs20242025  = COHBroadDfs20242025.dropna(subset=[col])

dfs = [
    CHCStrictDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'CHCStrictNumWalkable'}),
    CHCBroadDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'CHCBroadNumWalkable'}),
    COHStrictDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'COHStrictNumWalkable'}),
    COHBroadDfs20182019[['URN', 'numWalkableCharities']].rename(columns={'numWalkableCharities': 'COHBroadNumWalkable'}),
]
borda = reduce(lambda a, b: a.merge(b, on='URN', how='inner'), dfs)

count_cols = ['CHCStrictNumWalkable', 'CHCBroadNumWalkable', 'COHStrictNumWalkable', 'COHBroadNumWalkable']

borda[count_cols] = borda[count_cols].fillna(0)

for c in count_cols:
    borda[f'{c}_rank'] = borda[c].rank(ascending=False, method='min')

rank_cols = [f'{c}_rank' for c in count_cols]
borda['borda_total'] = borda[rank_cols].sum(axis=1)
borda['borda_rank'] = borda['borda_total'].rank(ascending=True, method='min')

borda = borda.merge(CHCStrictDfs20182019.drop(columns=['numWalkableCharities']), on='URN', how='left')

N = len(borda)
for c in count_cols:
    borda[f'{c}_points'] = N - borda[f'{c}_rank']

points_cols = [f'{c}_points' for c in count_cols]
borda['borda_points_total'] = borda[points_cols].sum(axis=1)

In [72]:
FINALDF = borda[['URN', 'EstablishmentName', 'StudentsAchievingGrade5PlusEngMathRate',
                 'borda_rank', 'borda_points_total', 'NumberOfPupils', 'PercentOfStudentReceivingFreeSchoolMeals',
                 'UrbanRuralClassification', 'GOR', 'LA (code)', 'LA (name)',
                 'LSOA (code)', 'MSOA (code)', 'Postcode',
                 'longitude', 'latitude', 'Easting', 'Northing']]
FINALDF = FINALDF.sort_values('borda_rank').reset_index(drop=True)
FINALDF.to_csv(f"Data/20242025/finalDf20242025.csv", index=False)

In [73]:
FINALDF

,URN,EstablishmentName,StudentsAchievingGrade5PlusEngMathRate,borda_rank,borda_points_total,NumberOfPupils,PercentOfStudentReceivingFreeSchoolMeals,UrbanRuralClassification,GOR,LA (code),LA (name),LSOA (code),MSOA (code),Postcode,longitude,latitude,Easting,Northing
0,100458.0,Central Foundation Boys' School,71.0,1.0,10029.0,135.0,59.0,Urban: Nearer to a major town or city,London,206,Islington,E01002704,E02000576,EC2A 4SH,-0.085823,51.525257,532888.0,182441.0
1,137789.0,Green Spring Academy Shoreditch,44.0,2.0,10023.0,176.0,68.0,Urban: Nearer to a major town or city,London,211,Tower Hamlets,E01004312,E02000872,E2 6NW,-0.069586,51.527392,534008.0,182708.0
2,100282.0,Our Lady's Convent Roman Catholic High School,49.0,3.0,10013.0,121.0,48.0,Urban: Nearer to a major town or city,London,204,Hackney,E01001809,E02000345,N16 5AF,-0.075207,51.574273,533481.0,187911.0
3,133599.0,Yesodey Hatorah Senior Girls School,68.0,4.0,10012.0,56.0,7.0,Urban: Nearer to a major town or city,London,204,Hackney,E01001826,E02000348,N16 6UB,-0.068341,51.573522,533959.0,187840.0
4,138202.0,Wapping High School,65.0,5.0,10008.0,37.0,68.0,Urban: Nearer to a major town or city,London,211,Tower Hamlets,E01004321,E02000884,E1 2DA,-0.060974,51.514975,534642.0,181343.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2504,136205.0,Wilmington Academy,38.0,1828.0,6905.0,160.0,18.0,Urban: Nearer to a major town or city,South East,886,Kent,E01024189,E02005038,DA2 7DR,0.192617,51.428068,552526.0,172176.0
2505,136638.0,Clyst Vale Community College,47.0,1828.0,6905.0,140.0,24.0,Larger rural: Nearer to a major town or city,South West,878,Devon,E01034620,E02007027,EX5 3AJ,-3.439868,50.758519,298538.0,96426.0
2506,135630.0,Longfield Academy,24.0,1828.0,6905.0,161.0,28.0,Urban: Nearer to a major town or city,South East,886,Kent,E01024158,E02005040,DA3 7PH,0.306792,51.396468,560573.0,168906.0
2507,135372.0,New Line Learning Academy,15.0,1828.0,6905.0,112.0,42.0,Urban: Nearer to a major town or city,South East,886,Kent,E01024404,E02005079,ME15 9QL,0.532142,51.249236,576813.0,153053.0
